# Official HEC-HMS Guide Mirror: Advanced Analysis Types

Official guide: https://www.hec.usace.army.mil/confluence/hmsdocs/hmsguides/advanced-analysis-types-in-hms

This notebook inventories the advanced-analysis artifacts present in the HMS sample project and demonstrates hms-commander scaffolding for batch runs and parameter ensembles. Native HMS optimization, uncertainty, ensemble, save-state, and depth-area analysis object APIs are tracked separately.

In [1]:
from pathlib import Path
import logging

import pandas as pd

logging.disable(logging.CRITICAL)


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "hms_commander").is_dir() and (candidate / "examples").is_dir():
            return candidate
    return start


REPO_ROOT = find_repo_root()
WORK_ROOT = REPO_ROOT / "examples" / "working" / "clb238_guides"
WORK_ROOT.mkdir(parents=True, exist_ok=True)
HMS_VERSION = "4.13"


from hms_commander import HmsExamples, HmsPrj

available_versions = HmsExamples.list_versions()
if HMS_VERSION not in available_versions:
    HMS_VERSION = available_versions[0]
HMS_EXE = HmsExamples.get_hms_exe(HMS_VERSION)


def init_sample_project(project_name, notebook_key):
    project_path = HmsExamples.extract_project(
        project_name,
        version=HMS_VERSION,
        output_path=WORK_ROOT / notebook_key,
        overwrite=True,
    )
    project = HmsPrj()
    project.initialize(project_path, hms_exe_path=HMS_EXE)
    return project, project_path

In [2]:
from hms_commander import HmsJython

project, project_path = init_sample_project("castro", "27_advanced_analysis")
advanced_dirs = ["optimizer", "ensemble", "forecast", "frequency", "montecarlo"]
advanced_inventory = pd.DataFrame([
    {
        "artifact_family": dirname,
        "exists": (project_path / dirname).exists(),
        "file_count": len([path for path in (project_path / dirname).rglob("*") if path.is_file()]) if (project_path / dirname).exists() else 0,
    }
    for dirname in advanced_dirs
])
advanced_inventory

,artifact_family,exists,file_count
0,optimizer,True,0
1,ensemble,True,0
2,forecast,True,0
3,frequency,True,0
4,montecarlo,True,0


In [3]:
run_names = project.list_run_names()
batch_script = HmsJython.generate_batch_compute_script(project_path, run_names)
batch_script_checks = pd.DataFrame([
    {"check": f"contains run {run_name}", "passed": run_name in batch_script}
    for run_name in run_names
] + [
    {"check": "contains computation summary", "passed": "Computation Summary" in batch_script},
])
assert batch_script_checks["passed"].all()
batch_script_checks

,check,passed
0,contains run Current,True
1,contains run Future,True
2,contains computation summary,True


In [4]:
parameter_ensemble = pd.DataFrame([
    {"member": "low_loss_fast_response", "curve_number_delta": -5, "lag_multiplier": 0.80},
    {"member": "baseline", "curve_number_delta": 0, "lag_multiplier": 1.00},
    {"member": "high_loss_slow_response", "curve_number_delta": 5, "lag_multiplier": 1.20},
])
parameter_ensemble["run_count"] = len(run_names)
parameter_ensemble

,member,curve_number_delta,lag_multiplier,run_count
0,low_loss_fast_response,-5,0.8,2
1,baseline,0,1.0,2
2,high_loss_slow_response,5,1.2,2


In [5]:
save_state_candidates = project.run_df[["name", "save_state_type", "time_series_output"]].copy()
save_state_candidates = save_state_candidates.rename(columns={"name": "run_name"})
save_state_candidates

,run_name,save_state_type,time_series_output
0,Current,None,Save All
1,Future,None,Save All


## Coverage Notes

The official advanced-analysis category includes optimization trials, uncertainty simulations, ensemble simulations, save states, and depth-area analysis. This notebook records the currently executable hms-commander pieces and points the native analysis-object work to CLB-290.